# Steel Slab Logistics Optimization: RL vs. MIP

This notebook runs the scheduling pipeline for the Steel Slab Logistics environment. Slabs arrive stochastically at a steel plant, cooling continuously over time. They must be scheduled to move from the yard into hot-rooms or under mobile thermal covers via a shared crane to slow down cooling. 

We solve this using two main techniques:
1. **Reinforcement Learning (RL):** Online decision-making using Proximal Policy Optimization (PPO) via `stable-baselines3`.
2. **Integer Programming (MIP):** Offline optimal benchmark scheduling using Google OR-Tools (CP-SAT backend) to compute a theoretical performance upper bound.

In [1]:
from utils.environment import SteelLogistics
from utils.lp import solve_episode_mip
import config

from torch import cuda
import pandas as pd

# RL Packages
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## 1. Global Setup and Parameters

We define our key flags and random seeds below:
* `TRAIN`: When set to `True`, trains a new PPO model. When `False`, loads the saved model weights.
* `USE_GPU`: Controls hardware acceleration. For this simulation, training on CPU is usually faster due to environment stepping overhead.
* `SEED`: Ensures reproducibility of the environment generation and policy evaluation.

In [2]:
TRAIN = True
USE_GPU = False # The RL training is faster with CPU
SEED = 42

In [3]:
# Verify GPU availability
if cuda.is_available() and USE_GPU:
    device = "cuda"
    print(f"GPU Name: {cuda.get_device_name(0)}")
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: cpu


## 2. Environment Initialization

We instantiate our custom Gymnasium environment `SteelLogistics` using configuration parameters imported from `config.py`. 

We then run `check_env(env, warn=True)` to validate that our action spaces, observation spaces, reset mechanisms, and step structures strictly conform to standard Gymnasium protocols.

In [4]:
config.time_frame = config.train_time_frame

# 1. Initialize your custom Gymnasium environment
env = SteelLogistics(config)

# 2. Validate the environment architecture
check_env(env, warn=True)

## 3. RL Agent Training

If `TRAIN` is `True`, we initialize a Stable Baselines3 **PPO (Proximal Policy Optimization)** agent with a Multi-Layer Perceptron (`MlpPolicy`) and run the model learning phase over `config.train_step_count` steps. 

Once trained, the frozen weights are saved to disk as `ppo_steel_logistics_model.zip`. If training is bypassed, we load the pre-trained weights instead.

In [5]:
if TRAIN:
    # 3. Initialize the PPO Agent
    model = PPO("MlpPolicy", env, verbose=1, device=device, tensorboard_log="./ppo_steel_tensorboard/")

    # 4. Train the Agent
    # 500,000 steps is a solid baseline for a simulation of this complexity
    # model.learn(total_timesteps=500000)
    model.learn(total_timesteps=config.train_step_count)

    # 6. Save the frozen weights for the final evaluation
    model.save("ppo_steel_logistics_model")

elif not TRAIN:
    # 5. Load the trained model for evaluation
    model = PPO.load("ppo_steel_logistics_model",env = env, device=device)

else:
    raise ValueError("TRAIN variable must be set to either True or False.")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ./ppo_steel_tensorboard/PPO_3
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 100       |
|    ep_rew_mean     | -7.81e+04 |
| time/              |           |
|    fps             | 1635      |
|    iterations      | 1         |
|    time_elapsed    | 1         |
|    total_timesteps | 2048      |
----------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 100           |
|    ep_rew_mean          | -7.72e+04     |
| time/                   |               |
|    fps                  | 1245          |
|    iterations           | 2             |
|    time_elapsed         | 3             |
|    total_timesteps      | 4096          |
| train/                  |               |
|    approx_kl            | 0.00031882265 |
|    clip_fraction        | 0  

## 4. Model Evaluation

With our PPO model loaded or trained, we evaluate its online scheduling performance. Using the `evaluate` method of our Gymnasium environment, we run test episodes over the test timeframe. 

The evaluation returns a combined history of all processed slabs and a list of step histories for each evaluation episode.

In [6]:
# --- Evaluation / Test Run ---
# Note: eval function now returns the combined DataFrame AND a list of individual episode DataFrames!
df_combined_history, episode_dfs = env.evaluate(
    model, 
    test_time_frame=config.test_time_frame, 
    total_timesteps=config.test_step_count,
    seed=SEED
)

print("--- Evaluation Completed ---")
print(f"Total Unique Slabs Generated: {len(df_combined_history)}")

# Display the arrival history DataFrame with 'episode' and 'seed' tags
df_combined_history.head(15)

--- Evaluation Completed ---
Total Unique Slabs Generated: 147


,step,id,initial_wait_time,initial_temp,seed,episode
0,0,0,7,915.009024,42,0
1,0,1,6,918.811294,42,0
2,2,2,7,893.675148,42,0
3,2,3,7,899.663977,42,0
4,3,4,6,915.555839,42,0
5,4,5,5,922.544824,42,0
6,5,6,6,907.375016,42,0
7,7,7,5,899.001482,42,0
8,7,8,4,886.381409,42,0
9,8,9,7,896.909410,42,0


## 5. Comparative Performance Analysis: Online RL vs. Offline MIP Baseline

To understand the effectiveness of our online RL agent, we compare its scoring against a mathematical upper bound benchmark. 

For each individual test episode:
1. **MIP Optimal (Google OR-Tools CP-SAT):** We solve the exact same sequence of arriving slabs using a Mixed-Integer Programming formulation. This offline baseline possesses *perfect information/foresight* about all future slab arrivals, computing the theoretical maximum score.
2. **RL Agent (PPO):** We step the environment deterministically using the same episode seed and sum up the rewards collected in real-time under zero-foresight online conditions.
3. **RL Gap (%):** We calculate the gap percentage to evaluate how close the RL agent gets to the theoretical absolute optimum.

In [7]:
print("--- RL vs. MIP Optimal Comparison ---")

# Define a list to collect results for the comparison table
comparison_data = []

for ep_idx, df_ep in enumerate(episode_dfs):
    # 1. Identify the seed used for this episode
    current_seed = df_ep['seed'].iloc[0] if not df_ep.empty else "N/A"
    
    # 2. Solve the corresponding MIP schedule
    mip_result = solve_episode_mip(df_ep, config)
    
    # 3. Calculate RL score by stepping the environment deterministically with the same seed
    obs, _ = env.reset(seed=int(current_seed))
    env.time_frame = config.test_time_frame
    
    terminated = False
    truncated = False
    current_episode_rl_reward = 0.0
    
    while not (terminated or truncated):
        action, _ = model.predict(obs, deterministic=True)
        
        # Trigger drain mode at the end of the active generation period to deliver all slabs
        if env.current_step >= env.time_frame:
            env.drain_mode = True
            
        obs, reward, terminated, truncated, _ = env.step(action)
        current_episode_rl_reward += reward

    comparison_data.append({
        "Episode": f"{ep_idx}",
        "Seed": current_seed,
        "Total Slabs": len(df_ep),
        "RL Score": current_episode_rl_reward,
        "MIP Optimal": mip_result['objective'] if mip_result else None,
        "MIP Status": mip_result['status'] if mip_result else "Infeasible",
        "MIP Failed Slabs": int(mip_result['failed_slabs']) if mip_result and mip_result['failed_slabs'] is not None else 0
    })
    
    mip_score_str = f"{mip_result['objective']:.2f}" if mip_result and mip_result['objective'] is not None else 'N/A'
    print(f"Solved Episode {ep_idx} - RL: {current_episode_rl_reward:.2f} | MIP: {mip_score_str}")

# Convert to a DataFrame for visualization
df_comparison = pd.DataFrame(comparison_data)

# Compute performance gap
df_comparison['RL Gap (%)'] = ((df_comparison['MIP Optimal'] - df_comparison['RL Score']) / df_comparison['MIP Optimal']) * 100

# Format and display
display(df_comparison.style.format({
    "RL Score": "{:.2f}",
    "MIP Optimal": "{:.2f}",
    "RL Gap (%)": "{:+.2f}%"
}).background_gradient(subset=["RL Score", "MIP Optimal"], cmap="Blues"))

--- RL vs. MIP Optimal Comparison ---
Solved Episode 0 - RL: 2424.34 | MIP: 8008.04
Solved Episode 1 - RL: 1020.31 | MIP: 9721.47
Solved Episode 2 - RL: 2238.95 | MIP: 9059.78
Solved Episode 3 - RL: 1393.63 | MIP: 7407.87
Solved Episode 4 - RL: 1407.00 | MIP: 11144.38
Solved Episode 5 - RL: 2401.80 | MIP: 8584.95
Solved Episode 6 - RL: 2124.33 | MIP: 10249.59
Solved Episode 7 - RL: 1092.64 | MIP: 6648.15
Solved Episode 8 - RL: 2385.77 | MIP: 9081.00
Solved Episode 9 - RL: 223.23 | MIP: 8441.32


,Episode,Seed,Total Slabs,RL Score,MIP Optimal,MIP Status,MIP Failed Slabs,RL Gap (%)
0,0,42,14,2424.34,8008.04,Optimal,3,+69.73%
1,1,43,15,1020.31,9721.47,Optimal,2,+89.50%
2,2,44,13,2238.95,9059.78,Optimal,1,+75.29%
3,3,45,14,1393.63,7407.87,Optimal,3,+81.19%
4,4,46,17,1407.00,11144.38,Optimal,2,+87.37%
5,5,47,18,2401.80,8584.95,Feasible,5,+72.02%
6,6,48,14,2124.33,10249.59,Optimal,1,+79.27%
7,7,49,14,1092.64,6648.15,Optimal,4,+83.56%
8,8,50,13,2385.77,9081.00,Optimal,1,+73.73%
9,9,51,15,223.23,8441.32,Optimal,3,+97.36%
